In [ ]:
# Lab type: review
# Course: AI403 — Building Production AI Agents
# Lesson: Single Agent vs. Multi-Agent: A Decision Procedure
# Task: A team says their multi-agent system beats a single agent by 13 points. Review the
# comparison, rerun it at an equal token budget, and decide where each design belongs.

# Lab: Is the Second Agent Earning Its Cost?

Both systems are simulated with fixed success rates per kind of case, so results are exactly reproducible. What you're reviewing is the *comparison* — the part teams get wrong.

**Outputs are cleared.** Run every cell top to bottom.

## Setup

In [ ]:
# No installs needed: this lab uses the Python standard library only.

In [ ]:
from collections import namedtuple

Case = namedtuple("Case", "id kind")
Run = namedtuple("Run", "passed tokens_used")

# 60 labelled cases. "sequential": each step depends on the last (refund decisions).
# "parallel": independent sub-questions (e.g. compare five vendors' contract terms).
CASES = ([Case(f"seq-{i:02d}", "sequential") for i in range(1, 41)] +
         [Case(f"par-{i:02d}", "parallel") for i in range(1, 21)])

TOKENS_PER_SAMPLE = 6000

def _draw(case, k=0, salt=0.0):
    """Deterministic, evenly spread pseudo-randomness, so pass rates match their probabilities."""
    i = int(case.id[-2:])
    return (i * 0.6180339 + k * 0.4142136 + salt) % 1.0

def run_multi(case):
    """Planner + researchers + writer. Parallel cases split cleanly; sequential ones lose facts in handoffs."""
    p, cost = (0.90, 30000) if case.kind == "parallel" else (0.60, 18000)
    return Run(_draw(case, salt=0.37) < p, cost)

def run_single(case, token_budget):
    """One agent. It spends its budget on independent samples and takes a majority vote."""
    p = 0.62 if case.kind == "sequential" else 0.45
    n = max(1, token_budget // TOKENS_PER_SAMPLE)
    votes = sum(_draw(case, k) < p for k in range(n))
    return Run(votes > n / 2, n * TOKENS_PER_SAMPLE)

def report(rows):
    for kind in ("sequential", "parallel", "all"):
        sel = [r for r in rows if kind == "all" or r["kind"] == kind]
        m = sum(r["multi_pass"] for r in sel) / len(sel)
        s = sum(r["single_pass"] for r in sel) / len(sel)
        mt = sum(r["multi_tokens"] for r in sel) / len(sel)
        st = sum(r["single_tokens"] for r in sel) / len(sel)
        print(f"{kind:<11} multi {m:.0%} @ {mt:,.0f} tok | single {s:.0%} @ {st:,.0f} tok")

print(len(CASES), "labelled cases")

## The team's comparison

In [ ]:
# --- THE TEAM'S COMPARISON (review this code — is it correct?) ---
rows = []
for case in CASES:
    multi = run_multi(case)
    single = run_single(case, token_budget=6000)          # "the usual single-agent setup"
    rows.append({"kind": case.kind, "multi_pass": multi.passed, "single_pass": single.passed,
                 "multi_tokens": multi.tokens_used, "single_tokens": single.tokens_used})
report(rows)

**Question 1.** The team reports "multi-agent: 70%, single agent: 57%". What's confounded in this comparison, and what does the token column already tell you about the sequential cases?

In [ ]:
# Your answer (as comments):

<details>
<summary>🔑 Reveal answer — Question 1</summary>

The multi-agent system spends on average 22,000 tokens per case; the single agent is capped at 6,000. The comparison measures architecture *plus* 3–5x more compute, so it can't tell you which one made the difference — the confound the 2026 studies in the lesson set out to remove.

The token column already undercuts the headline on sequential cases: the single agent reaches 62% on 6,000 tokens, the multi-agent system 60% on 18,000. Same quality at a third of the cost, before any matching.

</details>

**Question 2.** Write `budget_matched(cases)` that gives the single agent exactly the tokens the multi-agent run used *on that case* (`multi.tokens_used`) and returns the same row format. Report it.

In [ ]:
# Work here: budget_matched

<details>
<summary>🔑 Reveal answer — Question 2</summary>

```python
def budget_matched(cases):
    rows = []
    for case in cases:
        multi = run_multi(case)
        single = run_single(case, token_budget=multi.tokens_used)
        rows.append({"kind": case.kind, "multi_pass": multi.passed, "single_pass": single.passed,
                     "multi_tokens": multi.tokens_used, "single_tokens": single.tokens_used})
    return rows

report(budget_matched(CASES))
```

</details>

**Question 3.** Read the budget-matched result by kind of case. Where does the second agent earn its cost, where does it lose, and why does giving the single agent *more* samples make it worse on parallel cases?

In [ ]:
# Your answer (as comments):

<details>
<summary>🔑 Reveal answer — Question 3</summary>

- **Sequential cases:** at an equal budget the single agent wins, 80% to 60%. The multi-agent system's handoffs lose facts; the single agent spends the same budget on independent attempts and a vote. This matches the lesson's evidence: on sequential work, multi-agent variants degrade performance.
- **Parallel cases:** the multi-agent system wins clearly, 90% to 25%. Independent sub-questions split cleanly, and each researcher's context stays focused.
- **Why more samples hurt:** majority voting helps only when each sample is right more often than not. On parallel cases the single agent's per-sample success is 45%, so voting over more samples converges on the *wrong* answer. Extra compute can't fix a design mismatch.

So the answer isn't "multi-agent" or "single agent" but routing: parallel cases to the multi-agent system, sequential ones to a single agent.

</details>

**Question 4.** Build that routed system: sequential cases go to the single agent with an 18,000 token budget, parallel cases to the multi-agent system. Report its pass rate and average tokens, and compare with each system alone.

In [ ]:
# Work here: routed system

<details>
<summary>🔑 Reveal answer — Question 4</summary>

```python
passed, spent = 0, 0
for case in CASES:
    run = run_multi(case) if case.kind == "parallel" else run_single(case, token_budget=18000)
    passed += run.passed
    spent += run.tokens_used
print(f"routed: {passed / len(CASES):.0%} @ {spent / len(CASES):,.0f} tokens per case")
```

About 83% at the same average spend as the multi-agent system alone (70%). The router itself is a classification call with an independent evaluation — routing accuracy on labelled cases — not an agent.

</details>

**Question 5 (judgement).** The multi-agent system has these agents. Apply the lesson's test — *what would each be evaluated on by itself?* — and say which deserve to exist.

| Agent | What it does |
|---|---|
| Planner | Writes a plan the other agents follow |
| Researchers (one per sub-question) | Each answers one independent sub-question with sources |
| Critic | Reads the draft and suggests improvements |
| Writer | Produces the final answer |

In [ ]:
# Your answer (as comments):

<details>
<summary>🔑 Reveal answer — Question 5</summary>

- **Researchers — keep (for parallel cases).** Each has an independent evaluation: did it find the labelled facts for its own sub-question? This is the parallel-work case.
- **Writer — this is the agent** (or the single agent's final step): judged by the whole system's output.
- **Planner — fold in.** Its plan can only be judged by whether the final answer is right, so it's a step inside the agent, not a separate agent. (A deterministic splitter of sub-questions may be code.)
- **Critic — only if it becomes a verifier.** "Suggests improvements" has no pass condition. A verifier that checks the draft's claims against sources can be evaluated by seeding known errors and counting how many it catches; then it earns its place.

</details>

## Summary

1. A fair comparison gives the single agent the _______ token budget the multi-agent run used.
2. Multi-agent designs earn their cost mainly on _______ work.
3. If you can't say what an agent would be evaluated on _______, it's a step, not an agent.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **same**
2. **parallel (independent)**
3. **by itself**

</details>